In [72]:
import pandas as pd
import os
import math

In [73]:
results_folder = "../../data/results/"
# school = "school_1"
# filename = "CP_20250429_121259.csv"
school = "school_2"
filename = "CP_20250429_121212.csv"
processed_data_folder = "../../data/processed_data/"

Help functions

In [74]:
def read_df(school, processed_data_folder, filename):
    path = os.path.join(processed_data_folder, school, filename)
    return pd.read_csv(path)

def read_group_preferences(school, processed_data_folder):
    df = read_df(school, processed_data_folder, 'group_preferences.csv')
    n_students, n_groups, min_group_size, max_extra_care_1, max_extra_care_2 = df.iloc[0]

    # Check if all values exist
    if pd.isnull(n_students) or pd.isnull(n_groups) or pd.isnull(min_group_size) or pd.isnull(max_extra_care_1) or pd.isnull(max_extra_care_2):
        raise ValueError("One or more values in group preferences are missing.")

    return n_students, n_groups, min_group_size, max_extra_care_1, max_extra_care_2

def get_max_group_size(min_group_size, n_students, n_groups):
    print(f'min_group_size: {min_group_size}, n_students: {n_students}, n_groups: {n_groups}')
    remaining = n_students - ( min_group_size * n_groups)
    max_size = min_group_size + (remaining // n_groups)
    if remaining % n_groups != 0:
        max_size += 1
    return max_size



In [75]:
class InputData:
    def __init__(self, group_preferences, info_students, info_teachers, constraints_students, constraints_teachers):
        self.group_preferences = group_preferences
        self.info_students = info_students
        self.info_teachers = info_teachers
        self.constraints_students = constraints_students
        self.constraints_teachers = constraints_teachers

class Groupvariables:
    def __init__(self, n_students, n_groups, min_group_size, max_extra_care_1, max_extra_care_2, max_group_size):
        self.n_students = n_students
        self.n_groups = n_groups
        self.min_group_size = min_group_size
        self.max_extra_care_1 = max_extra_care_1
        self.max_extra_care_2 = max_extra_care_2
        self.max_group_size = max_group_size

def read_dfs(school, processed_data_folder):
    return InputData(
        read_df(school, processed_data_folder, 'group_preferences.csv'),
        read_df(school, processed_data_folder, 'info_students.csv'),
        read_df(school, processed_data_folder, 'info_teachers.csv'),
        read_df(school, processed_data_folder, 'constraints_students.csv'),
        read_df(school, processed_data_folder, 'constraints_teachers.csv')
    )

def read_variables(data):
    group_preferences = data.group_preferences
    n_students, n_groups, min_group_size, max_extra_care_1, max_extra_care_2 = group_preferences.iloc[0]
    max_group_size = get_max_group_size(min_group_size, n_students, n_groups)
    print(f"Max group size: {max_group_size}")
    return Groupvariables(n_students, n_groups, min_group_size, max_extra_care_1, max_extra_care_2, max_group_size)

data = read_dfs(school, processed_data_folder)
variables = read_variables(data)

min_group_size: 24, n_students: 77, n_groups: 3
Max group size: 26


In [76]:


# Read in all necessary files
groups = pd.read_csv(os.path.join(results_folder, school, filename))

# Merge dataframes
merged = pd.merge(groups, data.info_students, on='Student', how='left')
merged.rename(columns={'Teacher': 'Assigned Group'}, inplace=True)

In [77]:
def get_total_preferences_satisfied(df):
    columns = ['Preference 1', 'Preference 2', 'Preference 3', 'Preference 4', 'Preference 5']
    total = 0

    for _, row in df.iterrows():
        student_group = row['Assigned Group']
        prefs = row[columns].tolist()
        # print(f'Student: {row["Student"]}, Assigned Group: {student_group}, Preferences: {prefs}')

        for pref_student in prefs:
            if pd.isna(pref_student):
                continue
            pref_group = df.loc[df['Student'] == pref_student, 'Assigned Group'].values[0]
            if pref_group == student_group:
                total += 1

    return total

In [78]:
def get_satisfied_preferences_per_student(df):
    columns = ['Preference 1', 'Preference 2', 'Preference 3', 'Preference 4', 'Preference 5']
    preferences = {}

    for _, row in df.iterrows():
        student = row['Student']
        student_group = row['Assigned Group']
        prefs = row[columns].tolist()

        count = 0
        for pref_student in prefs:
            if pd.isna(pref_student):
                continue
            pref_group = df.loc[df['Student'] == pref_student, 'Assigned Group'].values[0]
            if pref_group == student_group:
                count += 1

        preferences[student] = count

    print(f"Preferences per student: {preferences}")
    return preferences

def get_average_preferences(df):
    total_preferences = get_total_preferences_satisfied(df)
    total_students = len(df)
    average_preferences = total_preferences / total_students
    return average_preferences

def get_total_preferences_provided(df):
    columns = ['Preference 1', 'Preference 2', 'Preference 3', 'Preference 4', 'Preference 5']
    total_preferences = 0

    for _, row in df.iterrows():
        preferences_provided = row[columns].notna().sum()
        total_preferences += preferences_provided

    return total_preferences

def get_satisfaction_rate(df):
    provided = get_total_preferences_provided(df)
    satisfied = get_total_preferences_satisfied(df)
    return satisfied / provided


In [ ]:
# TODO checken hoe goed dit werkt
def get_minimum_preferences_satisfied(df):
    columns = ['Preference 1', 'Preference 2', 'Preference 3', 'Preference 4', 'Preference 5']
    min_satisfied = float('inf')

    for _, row in df.iterrows():
        student = row['Student']
        student_group = row['Assigned Group']
        prefs = row[columns].tolist()

        # Count how many preferences were provided
        prefs_provided = [p for p in prefs if not pd.isna(p)]
        n_provided = len(prefs_provided)

        if n_provided == 0:
            continue  # Skip students with no preferences

        # Count how many preferences were satisfied
        n_satisfied = 0
        for pref_student in prefs_provided:
            pref_group = df.loc[df['Student'] == pref_student, 'Assigned Group'].values[0]
            if pref_group == student_group:
                n_satisfied += 1

        # Update minimum satisfied if it's lower
        min_satisfied = min(min_satisfied, n_satisfied)

    return min_satisfied if min_satisfied != float('inf') else 0

Preferences per student: {'Lieve': 5, 'Liam C': 5, 'Kiyan': 0, 'Quinn P': 5, 'Vic': 5, 'Quinn K': 5, 'Davey': 5, 'Thalita': 4, 'Inoshan': 5, 'Felyne': 1, 'Mace': 4, 'Liam B': 5, 'Joppe': 4, 'Steffi': 4, 'Soufian': 5, 'Mattheus': 4, 'Aimee': 4, 'Mairo': 5, 'Zayleah': 4, 'Jalen': 4, 'Malia': 4, 'Tess': 4, 'Loïs': 4, 'Isa-Mea': 5, 'Floor': 5, 'Emma': 3, 'Armin': 5, 'Julian': 5, 'Tirra': 3, 'Mariastella': 4, 'Stijn': 3, 'Enola': 4, 'Cas': 4, 'Alex 5': 2, 'Yasmeem': 0, 'IJsbrand': 0, 'Sofia': 4, 'Delano': 4, 'Elia': 3, 'Foss': 4, 'Nora 5': 5, 'Thijmen': 5, 'Ryan': 4, 'Nora 4': 2, 'Coco': 4, 'Adam': 5, 'Keyan': 4, 'Fedde': 3, 'Alex 4': 5, 'Daan': 2, 'Guusje': 4, 'Youssif': 0, 'Lucas': 0, 'Brenden': 5, 'Jenairo': 5, 'Joël': 5, 'Jip': 5, 'Aeden': 4, 'Ida': 4, 'Jaxx': 5, 'Dimas': 3, 'Macey': 5, 'Benthe': 5, 'Rafael': 5, 'Thomas': 5, 'Dilayla': 5, 'Diana': 5, 'Morgan': 4, 'Jahmaira': 5, 'Eowyn': 5, 'Bram': 5, 'Liv': 5, 'Alicia': 5, 'Jake': 5, 'Rayan': 5, 'Zoë': 5, 'Nova': 4}


0

In [80]:
from check_constraints import run_check_constraints


def main(data, variables):
    # CHECK CONSTRAINTS
    # Check if all groups satisfy the constraints
    groups = data.info_teachers['Teacher'].tolist()
    if run_check_constraints(groups, merged, data, variables):
        print("All constraints are satisfied.")

    # SOLUTION QUALITY
    # Total preferences satisfied
    total_preferences = get_total_preferences_satisfied(merged)
    print(f"Total preferences satisfied: {total_preferences}")

    # Avg preferences satisfied
    avg_preferences = get_average_preferences(merged)
    print(f"Avg preferences satisfied: {avg_preferences:.2f}")

    # Satisfaction rate
    satisfaction_rate = get_satisfaction_rate(merged)
    print(f"Satisfaction rate: {satisfaction_rate:.2f}")

    # Min preferences satisfied for all students with that many preferences





    # EXTRA SOLUTION QUALITY
    # Optimal solution (number of preferences provided by students) / or optimal solution?
    n_provided_preferences = get_total_preferences_provided(merged)
    print(f"Total preferences provided by all students: {n_provided_preferences}")

    # Absolute difference between preferences satisfied and preferences given

    # Relative excess of preferences satisfied

    # --- fixed cost?

main(data, variables)

All constraints are satisfied.
Total preferences satisfied: 310
Avg preferences satisfied: 4.03
Satisfaction rate: 0.83
Total preferences provided by all students: 375


In [ ]:

# def evaluate_accuracy_from_csv(assigned_groups, preferences_df):
#     total_points = 0
#     student_groups = assigned_groups.set_index('ID')['Assigned_Group'].to_dict()

#     for _, row in preferences_df.iterrows():
#         student_id = row['ID']
#         preferences = row[['Preference_1', 'Preference_2', 'Preference_3', 'Preference_4', 'Preference_5']].dropna().tolist()

#         assigned_group = student_groups.get(student_id)

#         student_score = 0
#         for i, preferred_student in enumerate(preferences):
#             if preferred_student in student_groups and student_groups[preferred_student] == assigned_group:
#                 student_score += (5 - i)

#         total_points += student_score

#     optimal_score = calculate_optimal_score(preferences_df)

#     if optimal_score == 0:
#         relative_excess = 0
#     else:
#         relative_excess = (optimal_score - total_points) / optimal_score * 100

#     absolute_difference = optimal_score - total_points
#     return total_points, optimal_score, relative_excess, absolute_difference


# # Evaluate the accuracy
# total_points, optimal_score, relative_excess, absolute_difference = evaluate_accuracy_from_csv(assigned_groups, preferences)

# print("Total Points:", total_points)
# print("Optimal Score:", optimal_score)
# print("Absolute Difference:", absolute_difference)
# print("Relative Excess (%):", relative_excess)